# Introduction to Deep Learning

## Lab 2: CNNs


This lab aims to familiarize you with the fundamental principles of Deep Learning for computer vision. There are two parts in this lab:

- Development of a convolutional neural network for image classification and optimization of its parameters.
- Transfer learning techniques, which are particularly useful when you have a limited amount of data.

## 1- CNNs for digit recognition


---


---

### MNIST Dataset

The MNIST Dataset is a collection of 70,000 black-and-white images representing handwritten digits from zero to nine, including 60,000 images for training and 10,000 images for testing. Each image has a resolution of 28x28 pixels and features a handwritten digit. This dataset is commonly used for image classification tasks.

<img src='https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png'>

### Python libraries

In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import seaborn as sns
import time
from sklearn.metrics import confusion_matrix
from typing import *
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Fixing the seed to get reproducible results

# To know more: https://pytorch.org/docs/stable/notes/randomness.html

def seed_it_all(seed: int = 7777) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_it_all()

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

batch_size_train = 64
batch_size_test = 1024
learning_rate = 0.001
num_epochs = 5
patience = 10

In [ ]:
# ---- Helpers (added for the answers): parameter counting and summary of every experiment
import json

def count_parameters(model, trainable_only=True):
    """Number of (trainable) parameters of a model."""
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

results = {}   # summary of every trained architecture (used to answer Q10 and Q15)

def record(name, model, history, **extra):
    """Store the main numbers of a training run."""
    val_acc = [float(a) for a in history['val_acc']]
    results[name] = dict(params=count_parameters(model),
                         best_val_acc=max(val_acc), best_epoch=val_acc.index(max(val_acc)) + 1,
                         final_train_acc=float(history['acc'][-1]), final_val_acc=val_acc[-1],
                         final_train_loss=float(history['loss'][-1]), final_val_loss=float(history['val_loss'][-1]),
                         epochs=len(val_acc), **extra)
    return results[name]


**Q1.**
Apply the following transformations to the training set images for preprocessing and data augmentation:

- RandomHorizontalFlip: Horizontally flips a given image with a probability of 0.5.
- RandomRotation: Randomly rotates a given image by an angle ranging from -10 to 10 degrees.
- RandomAffine: Applies a random affine transformation. Set degrees to 0, shear to 10, and scale between 0.8 and 1.2.
- ColorJitter: Randomly modifies the brightness, contrast, and saturation of an image. Set brightness to 0.2, contrast to 0.2, and saturation to 0.2.
- ToTensor: Converts images initially in PIL format to PyTorch tensors.
- Normalize: Normalizes images using a mean of 0.1307 and a standard deviation of 0.3081.

You can find the necessary information on the following pages:

    https://pytorch.org/vision/0.15/transforms.html
    https://iq.opengenus.org/data-augmentation/

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),                          # horizontal flip with probability 0.5
    transforms.RandomRotation(degrees=10),                           # random rotation in [-10, +10] degrees
    transforms.RandomAffine(degrees=0, shear=10, scale=(0.8, 1.2)),  # random shear (+-10 deg) and zoom (x0.8 to x1.2)
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # random photometric changes
    transforms.ToTensor(),                                           # PIL image -> float tensor in [0, 1], shape (1, 28, 28)
    transforms.Normalize((0.1307,), (0.3081,)),                      # (x - mean) / std
])

**Q2.**  Apply the relevant transformations to the testing set images

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),                      # same conversion as for training
    transforms.Normalize((0.1307,), (0.3081,)), # same normalization as for training (but NO random augmentation)
])

Loading training and test data with preprocessing and data augmentation.

In [ ]:
train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train, batch_size=batch_size_train, shuffle=True)

test = torchvision.datasets.MNIST(root='/files/', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test, batch_size=batch_size_test, shuffle=False)

dataloaders = {"train" : train_loader, "test" : test_loader}

In [ ]:
print("Training data dimension : {}".format(train.data.shape))
print("Training labels dimension : {}".format(train.targets.shape))
print("test data dimension : {}".format(test.data.shape))
print("Test labels dimension : {}".format(test.targets.shape))

### Visualization of Training Data

In [ ]:
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.tight_layout()
  plt.imshow(train.data[i], cmap='gray', interpolation='none')
  plt.title("Label : {}".format(train.targets[i]))
  plt.xticks([])
  plt.yticks([])
plt.show()


### Fully-Connected Layers

**Q3.** The following cell launches the training of the defined model. Take a moment to read it and understand the main blocks. Comment each block.

In [ ]:
# ---------------------------------------------------------------------------------------------
# BLOC 1 - Early stopping : on s'arrête si le meilleur score (accuracy) date d'au moins `patience` epochs
# ---------------------------------------------------------------------------------------------
def early_stopping(scores: List[float], patience: int):
    last_idx = len(scores) - 1
    idx_val = scores.index(max(scores))
    if abs(last_idx - idx_val) >= patience:
        print("Early stopping ! Did not improve in {} iterations".format(patience))
        return True
    else:
        pass

# ---------------------------------------------------------------------------------------------
# Fonction d'entraînement complète (modèle, perte, optimiseur, données, nombre d'epochs, patience)
# ---------------------------------------------------------------------------------------------
def train_model(model, criterion, optimizer, dataloaders, device, num_epochs, patience,
                nb_train: int=60000, nb_test: int=10000, print_every: int=1):

    print('Model=%s, opt=%s(lr=%f), epochs=%d, device=%s\n' % \
        (type(model).__name__, type(optimizer).__name__,
        optimizer.param_groups[0]['lr'], num_epochs, device))

    # BLOC 2 - Initialisation : chrono, copie des meilleurs poids, dictionnaire `history` (courbes loss/accuracy)
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    history = {}
    history['loss'] = []
    history['val_loss'] = []
    history['acc'] = []
    history['val_acc'] = []
    best_epoch_acc_val = 0

    # BLOC 3 - Boucle sur les epochs (une epoch = un passage complet sur le jeu d'entraînement ET sur le jeu de test)
    for epoch in range(1, num_epochs+1):

        loss_train_tmp, loss_test_tmp = [], []
        running_corrects_train, running_corrects_test = 0, 0

        # BLOC 4 - Deux phases par epoch : 'train' (on apprend) puis 'test' (on évalue seulement)
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()   # mode entraînement : dropout actif
            else:
                model.eval()    # mode évaluation : dropout désactivé

            # BLOC 5 - Boucle sur les mini-batchs fournis par le DataLoader
            for idx, (inputs, labels) in enumerate(dataloaders[phase]):

                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()   # remise à zéro des gradients (PyTorch les accumule)

                # BLOC 6 - Passe avant (les gradients ne sont calculés qu'en phase 'train')
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs).to(device)   # logits
                    _, preds = torch.max(outputs, 1)       # classe prédite = indice du plus grand logit

                    loss = criterion(outputs, labels)      # perte (entropie croisée) du mini-batch

                    if phase == 'train':
                        loss.backward()    # rétropropagation : gradients de la perte
                        optimizer.step()   # mise à jour des paramètres (Adam)
                        loss_train_tmp.append(loss.item())
                        running_corrects_train += torch.sum(preds == labels.data).cpu()

                if phase == 'test':
                    loss_test_tmp.append(loss.item())
                    running_corrects_test += torch.sum(preds == labels.data).cpu()

        # BLOC 7 - Fin d'epoch : moyennes des pertes et accuracies (en %) ajoutées à `history`
        history['loss'].append(np.average(loss_train_tmp))
        history['val_loss'].append(np.average(loss_test_tmp))

        history['acc'].append((running_corrects_train / nb_train) * 100)
        history['val_acc'].append((running_corrects_test / nb_test) * 100)

        # BLOC 8 - On garde les poids de la meilleure epoch (selon l'accuracy sur le jeu de test), on affiche, et on teste les arrêts
        if history['val_acc'][-1] > best_epoch_acc_val:
            best_epoch_acc_val = history['val_acc'][-1]
            best_model_wts = copy.deepcopy(model.state_dict())
        if epoch % print_every == 0:
            print("Epoch {}/{}, train loss: {:.3f}, test loss: {:.3f}, train acc: {:.3f}, val acc: {:.3f}, best val acc: {:.3f}".format(
                epoch, num_epochs, history['loss'][-1], history['val_loss'][-1], history['acc'][-1], history['val_acc'][-1], max(history['val_acc'])))
        if history['val_acc'][-1] == 100:
            print("Early stoping ! Top score achieved")
            break
        if early_stopping(history['val_acc'], patience):
            break

    # BLOC 9 - Fin d'entraînement : durée, rechargement des MEILLEURS poids
    time_elapsed = time.time() - since

    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))

    print("Best accuracy : {}".format(best_epoch_acc_val))

    model.load_state_dict(best_model_wts)

    model.eval()
    model.to(device)

    # BLOC 10 - Prédictions finales sur le jeu de test avec le meilleur modèle (pour la matrice de confusion)
    all_labels = []

    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(dataloaders["test"]):
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs).to(device)
            all_labels.extend(list(labels.to("cpu").numpy()))
            if idx == 0:
                _, preds = torch.max(outputs, 1)
            else:
                _, tmp_preds = torch.max(outputs, 1)
                preds = torch.cat((preds, tmp_preds), axis=0)

    return model, history, preds, np.array(all_labels)

**Q4.** Implement a fully connected neural network consisting of three hidden layers and an output layer with 1024, 1024, 512, and 10 neurons, respectively.

Below is a summary of the operations:

**FULLY-CONNECTED -> ReLU -> DROPOUT -> FULLY-CONNECTED -> ReLU -> DROPOUT -> FULLY-CONNECTED -> ReLU -> DROPOUT -> FULLY-CONNECTED**



In [ ]:
class FullyConnected(nn.Module):
    def __init__(self):
        super(FullyConnected, self).__init__()

        self.fc1 = nn.Linear(28*28, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, 10)
        self.dropout = nn.Dropout(0.2)   # dropout probability (0.2, as for the CNN of Q7)

    def forward(self, x):
        x = x.view(-1, 28*28)  # Flatten the input
        x = self.dropout(F.relu(self.fc1(x)))   # FC -> ReLU -> DROPOUT
        x = self.dropout(F.relu(self.fc2(x)))   # FC -> ReLU -> DROPOUT
        x = self.dropout(F.relu(self.fc3(x)))   # FC -> ReLU -> DROPOUT
        out = self.fc4(x)                       # FC (logits: no softmax, CrossEntropyLoss applies it)
        return out


**Q5.**: Compute the number of trainable parameters of the model (by hand and numerically).

In [ ]:
# Q5 (numerically): trainable parameters of the fully-connected network, layer by layer
fc_model = FullyConnected()
for name, parameter in fc_model.named_parameters():
    print(f"{name:11s} shape = {str(tuple(parameter.shape)):13s} -> {parameter.numel():>9,d} parameters")
print(f"\nTotal trainable parameters : {count_parameters(fc_model):,}")

**Q6.**: In the following cell, you will complete the missing parts:

* Define the appropriate loss function for the problem we are addressing.  
* Use the Adam optimization algorithm with the previously defined learning rate, whose value is `'learning_rate'`.


In [ ]:
model = FullyConnected()
criterion = nn.CrossEntropyLoss()                            # multi-class classification: softmax + negative log-likelihood
optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # Adam with the learning rate defined above (0.001)

In [ ]:
trained_model, history, preds, labels = train_model(model.to(device), criterion, optimizer, dataloaders, device, num_epochs, patience)

In [ ]:
history_fc = history                          # keep the curves of the fully-connected network
record("FC 1024-1024-512", model, history_fc, kind="FC")
print(results["FC 1024-1024-512"])

### Convolutional Neural Networks (CNNs)

**Q7.**: Implement a convolutional neural network consisting of two blocks, each including a convolutional layer, a non-linear ReLU activation function, a dropout layer, and a max-pooling layer. Then, add a fully connected layer with 128 neurons, followed by an output fully connected layer with 10 neurons.  

Below is a summary of the operations:

_CONV2D -> ReLU -> DROPOUT -> -> MAXPOOL -> CONV2D -> ReLU -> MAXPOOL -> DROPOUT -> FLATTEN -> FULLY-CONNECTED -> FULLY-CONNECTED_

You can find all the information related to the implementation of these layers on the following page:  
[https://pytorch.org/docs/stable/nn.html#convolution-layers](https://pytorch.org/docs/stable/nn.html#convolution-layers)


In [ ]:
# Define a simple CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)    # (1, 28, 28)  -> (32, 26, 26)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)   # (32, 13, 13) -> (64, 11, 11)
        self.pool = nn.MaxPool2d(2)                     # divides height and width by 2 (floor): 26 -> 13 and 11 -> 5
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)           # flatten size = 64 x 5 x 5 = 1600
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Conv2d : 32 filtres, filtre 3x3,  ReLU
        x = F.relu(self.conv1(x))
        # Dropout :probability = 0.2
        x = self.dropout(x)
        # Max Pooling : filter of size 2
        x = self.pool(x)
        # Conv2d : 64 filtres, filter 3x3, ReLU
        x = F.relu(self.conv2(x))
        # Dropout :probability = 0.2
        x = self.dropout(x)
        # Max Pooling : filter of size 2
        x = self.pool(x)
        # Flatten
        x = torch.flatten(x, 1)
        # Fully-connected (flatten size,128),  ReLU
        x = F.relu(self.fc1(x))
        # Fully-connected (128,10)
        out = self.fc2(x)
        return out


**Q8.**: In the following cell, you will complete the missing parts:

* Instantiate the model
* Define the appropriate loss function for the problem we are addressing.  
* Use the Adam optimization algorithm with the previously defined learning rate, whose value is `'learning_rate'`.


In [ ]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
total_params = 0
for _, parameter in model.named_parameters():
    if not parameter.requires_grad: continue
    param = parameter.numel()
    total_params += param
print("\nTotal Trainable Parameters : {}\n".format(total_params))

**Q9.**: Compute the number of trainable parameters of the model (by hand).

In [ ]:
# Q9 (check): shape of the tensor after every layer, for ONE 28x28 image
x = torch.zeros(1, 1, 28, 28)
cnn_check = CNN()
x = F.relu(cnn_check.conv1(x)); print("conv1 + ReLU :", tuple(x.shape))
x = cnn_check.pool(x);          print("max-pool     :", tuple(x.shape))
x = F.relu(cnn_check.conv2(x)); print("conv2 + ReLU :", tuple(x.shape))
x = cnn_check.pool(x);          print("max-pool     :", tuple(x.shape))
x = torch.flatten(x, 1);        print("flatten      :", tuple(x.shape))
print()
for name, parameter in cnn_check.named_parameters():
    print(f"{name:12s} shape = {str(tuple(parameter.shape)):16s} -> {parameter.numel():>8,d} parameters")

In the following cell, you will launch the training of the convolutional neural network defined above using 10 epochs and a batch size of 64.

In [ ]:
trained_model, history, preds, labels = train_model(model.to(device), criterion, optimizer, dataloaders, device, num_epochs, patience)

In [ ]:
history_cnn = history                         # keep the curves of the CNN
record("CNN 32-64 / fc 128", model, history_cnn, kind="CNN")
print(results["CNN 32-64 / fc 128"])

### Visualisation of the performances of the model on the training an test datasets

In [ ]:
plt.plot(history['loss'], label="Train loss")
plt.plot(history['val_loss'], label="Test loss")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Loss")
plt.title("Loss en fonction du nombre d'epoch")
plt.show()

In [ ]:
plt.plot(history['acc'], label="Train accuracy")
plt.plot(history['val_acc'], label="Test accuracy")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy en fonction du nombre d'epoch")
plt.show()

In [ ]:
cm = confusion_matrix(labels, preds.to('cpu').numpy())
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues')
plt.xlabel('Prediction')
plt.ylabel('Labels')
plt.title("Matrice de confusion top accuracy")
plt.show()

**Q10.**:

* First, report the results obtained above.  
* Then, vary the number of hidden layers as well as the number of neurons in them, and record the accuracy for each tested architecture. Additionally, detail the calculation of the total number of parameters associated with each architecture.



In [ ]:
# ---- Q10: architectures with a configurable number of hidden layers / neurons / filters
class MLP(nn.Module):
    """Fully-connected network with any list of hidden sizes. Hidden layer = FC -> ReLU -> Dropout(0.2)."""
    def __init__(self, hidden_sizes, p_drop=0.2):
        super().__init__()
        sizes = [28 * 28] + list(hidden_sizes) + [10]
        self.layers = nn.ModuleList([nn.Linear(a, b) for a, b in zip(sizes[:-1], sizes[1:])])
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        for layer in self.layers[:-1]:
            x = self.dropout(F.relu(layer(x)))
        return self.layers[-1](x)


class CNNVariant(nn.Module):
    """Blocks Conv(3x3) -> ReLU -> Dropout(0.2) -> MaxPool(2), then FC(fc) -> ReLU -> FC(10)."""
    def __init__(self, filters=(32, 64), fc=128, p_drop=0.2):
        super().__init__()
        chans = [1] + list(filters)
        self.convs = nn.ModuleList([nn.Conv2d(a, b, kernel_size=3) for a, b in zip(chans[:-1], chans[1:])])
        self.pool, self.dropout = nn.MaxPool2d(2), nn.Dropout(p_drop)
        size = 28
        for _ in filters:
            size = (size - 3 + 1) // 2          # 3x3 convolution without padding, then 2x2 max-pooling (floor)
        self.fc1 = nn.Linear(chans[-1] * size * size, fc)
        self.fc2 = nn.Linear(fc, 10)

    def forward(self, x):
        for conv in self.convs:
            x = self.pool(self.dropout(F.relu(conv(x))))
        x = torch.flatten(x, 1)
        return self.fc2(F.relu(self.fc1(x)))


def layer_details(model):
    """Hand-calculation of the parameters of every layer: inputs x outputs + biases."""
    rows = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            rows.append(f"{m.in_features}x{m.out_features}+{m.out_features}={m.weight.numel() + m.bias.numel():,}")
        elif isinstance(m, nn.Conv2d):
            k = m.kernel_size[0]
            rows.append(f"{m.out_channels}x{m.in_channels}x{k}x{k}+{m.out_channels}={m.weight.numel() + m.bias.numel():,}")
    return rows


def run_experiment(name, factory, **extra):
    """Train a fresh model (same seed, same optimizer, same number of epochs) and store its results."""
    seed_it_all()                                           # same seed for every experiment -> fair comparison
    model = factory().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    t0 = time.time()
    _, hist, _, _ = train_model(model, criterion, optimizer, dataloaders, device, num_epochs, patience)
    record(name, model, hist, seconds=time.time() - t0, details=layer_details(model), **extra)
    print(f"--> {name}: {count_parameters(model):,} parameters, best test accuracy = {results[name]['best_val_acc']:.2f} %\n")


In [ ]:
# Fully-connected networks: vary the number of hidden layers and of neurons
fc_architectures = {
    "FC 2048-1024-512-256": [2048, 1024, 512, 256],   # 4 hidden layers, wider
    "FC 512-256": [512, 256],                          # 2 hidden layers
    "FC 256": [256],                                   # 1 hidden layer
    "FC 64": [64],                                     # 1 small hidden layer
    "FC 32-16": [32, 16],                              # 2 very small hidden layers
}
for name, hidden in fc_architectures.items():
    run_experiment(name, lambda h=hidden: MLP(h), kind="FC")

In [ ]:
# Convolutional networks: vary the number of filters, of blocks and the size of the dense layer
cnn_architectures = {
    "CNN 16-32 / fc 64": dict(filters=(16, 32), fc=64),
    "CNN 64-128 / fc 256": dict(filters=(64, 128), fc=256),
    "CNN 32 (1 block) / fc 128": dict(filters=(32,), fc=128),
}
for name, cfg in cnn_architectures.items():
    run_experiment(name, lambda c=cfg: CNNVariant(**c), kind="CNN")

In [ ]:
# Summary of all MNIST experiments (same budget: Adam, lr=0.001, 5 epochs, batch size 64)
print(f"{'Architecture':28s} {'#parameters':>12s} {'best test acc':>14s} {'last train acc':>15s}  {'epoch':>5s}")
for name, r in sorted(results.items(), key=lambda kv: (kv[1]['kind'], kv[1]['params'])):
    if r['kind'] not in ('FC', 'CNN'):
        continue
    print(f"{name:28s} {r['params']:>12,d} {r['best_val_acc']:>13.2f}% {r['final_train_acc']:>14.2f}%  {r['best_epoch']:>5d}")

plt.figure(figsize=(7, 4.5))
for kind, color in (('FC', 'tab:blue'), ('CNN', 'tab:orange')):
    pts = [(r['params'], r['best_val_acc'], n) for n, r in results.items() if r['kind'] == kind]
    plt.scatter([p[0] for p in pts], [p[1] for p in pts], c=color, label=kind, s=60)
    for p in pts:
        plt.annotate(p[2].split(' ', 1)[1], (p[0], p[1]), fontsize=7, xytext=(4, 4), textcoords='offset points')
plt.xscale('log'); plt.xlabel('Number of trainable parameters (log scale)'); plt.ylabel('Best test accuracy (%)')
plt.title('MNIST: accuracy versus model size'); plt.grid(alpha=0.3); plt.legend(); plt.show()

## 2- Transfer Learning


---


---


Here, we will use the Transfer Learning technique to perform image classification. Indeed, it is rarely necessary to train a model from scratch, given the challenge of obtaining a sufficiently large dataset. Instead, it is common to use pre-trained models on large databases such as ImageNet.

There are two main procedures for Transfer Learning, as explained below:

* **Fine-tuning**: Instead of opting for random initialization, we initialize a pre-trained network for feature extraction, then add new fully connected layers.

* **Fixed feature extractor(Frozen Pre-Trained Model)**: We prevent the weights of the feature extraction part from updating, except for the newly added fully connected layers. Only the fully connected layer is trained.


**Q11.** Download the following dataset (https://download.pytorch.org/tutorial/hymenoptera_data.zip) and place it in a new folder named **data** in your drive.  

Here, we will focus on classifying ants and bees. To achieve this, we have a dataset consisting of **120 images** for each type of insect for training and **75 images** for validation/testing.


In [ ]:
# Mount Google Drive when running on Google Colab (skipped when running elsewhere)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running on Google Colab: Google Drive is not mounted, the data will be stored in ./data")

Replace the following path with your path

In [ ]:
if IN_COLAB:
    %cd /content/drive/'My Drive'/'Polytech'/'data'
else:
    # Outside Colab: download and unzip the hymenoptera dataset (ants / bees) in ./data
    import urllib.request, zipfile
    os.makedirs('data', exist_ok=True)
    os.chdir('data')
    if not os.path.isdir('hymenoptera_data'):
        urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip', 'hymenoptera_data.zip')
        zipfile.ZipFile('hymenoptera_data.zip').extractall('.')
        os.remove('hymenoptera_data.zip')

In [ ]:
! ls

In [ ]:
import torch.backends.cudnn as cudnn
from torchvision import datasets, models, transforms
from PIL import Image
from tempfile import TemporaryDirectory

In [ ]:
# Preprocessing and data augmentation

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
data_dir = 'hymenoptera_data'
dict_train_test ={'train':'train', 'test':'val'}
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[dict_train_test[x]], batch_size=4,
                                             shuffle=True, num_workers=4)
              for x in ['train', 'test']}
dataset_sizes = {x: len(image_datasets[dict_train_test[x]]) for x in ['train', 'test']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Visualizing a few datapoints

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)



inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs)

imshow(out, title=[class_names[x] for x in classes])

Function for visualizing predictions

In [ ]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['test']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title(f'predicted: {class_names[preds[j]]}')
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)


### Fine-tuning

**Q11.**: Set up a series of fully connected (FC) layers following the feature extraction part of the **ResNet-50** model.  

- The first layer will take as input the dimension of the last layer of **ResNet-18** and output **512 neurons**.  
- Then, there will be a second layer with **128 neurons**, followed by the output layer, which will have a number of neurons equal to the number of classes.  


In [ ]:
seed_it_all()   # reproducible initialization of the new fully-connected head

model_ft = models.resnet50(weights='IMAGENET1K_V1') # ResNet-50 pretrained on ImageNet
print(model_ft)

num_ftrs = model_ft.fc.in_features # dimension of the last layer or  ResNet-50  for the feature extraction

model_ft.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),            # 2048 -> 512
    nn.ReLU(),
    nn.Linear(512, 128),                 # 512 -> 128
    nn.ReLU(),
    nn.Linear(128, len(class_names))     # 128 -> number of classes (2: ants / bees)
    )


In [ ]:
# Number of trainable parameters of the fine-tuned network (all the layers are trained)
head = model_ft.fc
print(f"ResNet-50 feature extractor (everything except fc): {count_parameters(model_ft) - count_parameters(head):>11,d}")
print(f"New fully-connected head 2048-512-128-2           : {count_parameters(head):>11,d}")
print(f"Total trainable parameters                         : {count_parameters(model_ft):>11,d}")

**Q12.**:  
* Define the appropriate loss function for the problem we are addressing.  
* Additionally, use the Adam optimization algorithm with a learning rate that you consider appropriate.  


In [ ]:
model_ft = model_ft.to(device)
criterion = nn.CrossEntropyLoss()                          # 2-class classification (ants / bees)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=1e-4)  # small learning rate: the pretrained weights must only be adjusted, not destroyed


In the following cell, you will launch the training of the convolutional neural network defined above using **25 epochs**. The batch size is set to **4** as defined earlier.  


In [ ]:
trained_model, history, preds, labels = train_model(model_ft, criterion, optimizer_ft, dataloaders, device, 25, patience, dataset_sizes['train'], dataset_sizes['test'])

In [ ]:
history_ft, preds_ft, labels_ft = history, preds, labels   # keep the fine-tuning results (variables are overwritten below)
record("TL fine-tuning ResNet-50 (lr=1e-4)", model_ft, history_ft, kind="TL", scenario="fine-tuning", lr=1e-4)
print(results["TL fine-tuning ResNet-50 (lr=1e-4)"])

Visualize the predictions

In [ ]:
visualize_model(model_ft)

In [ ]:
plt.plot(history['loss'], label="Train loss")
plt.plot(history['val_loss'], label="Test loss")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Loss")
plt.title("Loss as a function of the number of epochs")
plt.show()

In [ ]:
plt.plot(history['acc'], label="Train accuracy")
plt.plot(history['val_acc'], label="Test accuracy")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy as a function of the number of epochs")
plt.show()

In [ ]:
cm = confusion_matrix(labels, preds.to('cpu').numpy())
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues')
plt.xlabel('Prediction')
plt.ylabel('Labels')
plt.title("Confusion matrix top accuracy")
plt.show()

### Fixed feature extractor(Frozen Pre-Trained Model)

**Q13.**: You will set up a series of fully connected (FC) layers following the feature extraction part of the **ResNet-50** model.  

- The first layer will take as input the dimension of the last layer of **ResNet-18** and output **512 neurons**.  
- Then, there will be a second layer with **128 neurons**, followed by the output layer, which will have a number of neurons equal to the number of classes.  

In [ ]:
seed_it_all()   # reproducible initialization of the new fully-connected head

model_conv = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_conv.parameters():
    param.requires_grad = False

# Parameters of newly constructed modules have requires_grad=True by default
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),            # 512 -> 512 (ResNet-18 outputs 512 features)
    nn.ReLU(),
    nn.Linear(512, 128),                 # 512 -> 128
    nn.ReLU(),
    nn.Linear(128, len(class_names)))    # 128 -> 2 classes

model_conv = model_conv.to(device)
criterion = nn.CrossEntropyLoss()
optimizer_conv = optim.Adam(model_conv.fc.parameters(), lr=1e-3)   # ONLY the new head is given to the optimizer

In [ ]:
# Frozen feature extractor: only the new head is trainable
n_total = count_parameters(model_conv, trainable_only=False)
n_train = count_parameters(model_conv)
print(f"Total parameters of the network : {n_total:>11,d}")
print(f"Trainable (new head 512-512-128-2): {n_train:>9,d}  ({100 * n_train / n_total:.1f} % of the network)")
print(f"Frozen (ResNet-18 backbone)      : {n_total - n_train:>11,d}")

**Q14.** What is the difference with the previous approach ?

In the following cell, you will launch the training of the convolutional neural network defined above using **25 epochs**. The batch size is set to **4** as defined earlier.  


In [ ]:
# NB: the original cell trained `model_ft` with `optimizer_ft` (the network of the previous section);
# here we train the frozen-backbone network `model_conv` with its own optimizer.
trained_model, history, preds, labels = train_model(model_conv, criterion, optimizer_conv, dataloaders, device, 25, patience, dataset_sizes['train'], dataset_sizes['test'])

In [ ]:
history_conv, preds_conv, labels_conv = history, preds, labels
record("TL frozen ResNet-18 (lr=1e-3)", model_conv, history_conv, kind="TL", scenario="frozen", lr=1e-3)
print(results["TL frozen ResNet-18 (lr=1e-3)"])

In [ ]:
plt.plot(history['loss'], label="Train loss")
plt.plot(history['val_loss'], label="Test loss")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Loss")
plt.title("Loss as a function of the number of epochs")
plt.show()

In [ ]:
plt.plot(history['acc'], label="Train accuracy")
plt.plot(history['val_acc'], label="Test accuracy")
leg = plt.legend(loc="best")
plt.xlabel("Nombre d'epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy as a function of the number of epochs")
plt.show()

**Q15.**:  
* First, show the results associated with the two types of transfer learning.  
* Vary the learning rate for both scenarios. 

In [ ]:
# ---- Q15: influence of the learning rate in both scenarios
EPOCHS_FT_VARIANT = 10       # fine-tuning ResNet-50 is slow on CPU: the extra learning rates are trained for 10 epochs
EPOCHS_FROZEN_VARIANT = 25   # frozen extractor: same budget as the main run (25 epochs)

def run_transfer(scenario, lr, epochs):
    """Fresh pretrained network (same seed) trained with Adam(lr) for `epochs` epochs."""
    seed_it_all()
    if scenario == "fine-tuning":
        m = models.resnet50(weights='IMAGENET1K_V1')
    else:
        m = models.resnet18(weights='IMAGENET1K_V1')
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Sequential(nn.Linear(m.fc.in_features, 512), nn.ReLU(), nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, len(class_names)))
    m = m.to(device)
    params = m.parameters() if scenario == "fine-tuning" else m.fc.parameters()
    _, hist, _, _ = train_model(m, nn.CrossEntropyLoss(), optim.Adam(params, lr=lr), dataloaders, device, epochs, patience,
                                dataset_sizes['train'], dataset_sizes['test'])
    record(f"TL {scenario} (lr={lr:g})", m, hist, kind="TL", scenario=scenario, lr=lr)
    return hist

tl_histories = {"fine-tuning": {1e-4: history_ft}, "frozen": {1e-3: history_conv}}   # main runs already done above
for lr in (1e-3, 1e-5):
    tl_histories["fine-tuning"][lr] = run_transfer("fine-tuning", lr, EPOCHS_FT_VARIANT)
for lr in (1e-2, 1e-4):
    tl_histories["frozen"][lr] = run_transfer("frozen", lr, EPOCHS_FROZEN_VARIANT)

In [ ]:
# Learning curves for every learning rate (left: fine-tuning ResNet-50, right: frozen ResNet-18)
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for col, scenario in enumerate(("fine-tuning", "frozen")):
    for lr, h in sorted(tl_histories[scenario].items()):
        axs[0, col].plot(range(1, len(h['val_acc']) + 1), [float(a) for a in h['val_acc']], label=f"lr={lr:g}")
        axs[1, col].plot(range(1, len(h['val_loss']) + 1), h['val_loss'], label=f"lr={lr:g}")
    axs[0, col].set_title(f"{scenario}: test accuracy"); axs[1, col].set_title(f"{scenario}: test loss")
    axs[0, col].set_ylabel("Accuracy (%)"); axs[1, col].set_ylabel("Loss")
    for row in (0, 1):
        axs[row, col].set_xlabel("Epoch"); axs[row, col].grid(alpha=0.3); axs[row, col].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Summary table of the transfer-learning experiments
print(f"{'Scenario':22s} {'lr':>8s} {'epochs':>6s} {'trainable params':>17s} {'best test acc':>14s} {'best epoch':>11s} {'last train acc':>15s}")
for name, r in sorted(results.items(), key=lambda kv: (kv[1].get('scenario', ''), kv[1].get('lr', 0))):
    if r['kind'] != 'TL':
        continue
    print(f"{r['scenario']:22s} {r['lr']:>8g} {r['epochs']:>6d} {r['params']:>17,d} {r['best_val_acc']:>13.2f}% {r['best_epoch']:>11d} {r['final_train_acc']:>14.2f}%")